<a href="https://colab.research.google.com/github/ayushhh026/RuppeRisk/blob/main/notebooks/Model09_Feature_Refinement.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Mount Drive and import required libraries
from google.colab import drive
drive.mount('/content/drive')

import os
import gc
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier

Mounted at /content/drive


In [2]:
# Define paths for Model08 artifacts
RESULTS_PATH = "/content/drive/MyDrive/RupeeRisk/"
MODEL08_DATA_PATH = RESULTS_PATH + "model08_consolidated_features.parquet"
MODEL08_FEATURE_LIST_PATH = RESULTS_PATH + "model08_feature_list.csv"
MODEL08_BENCHMARK_PATH = RESULTS_PATH + "model08_benchmark.csv"
os.makedirs(RESULTS_PATH, exist_ok=True)

In [3]:
# Load the fully-engineered feature table saved at the end of Model08
application_model = pd.read_parquet(MODEL08_DATA_PATH)
print("Model08 consolidated shape:", application_model.shape)
display(application_model.head())

Model08 consolidated shape: (307511, 402)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,CC_CC_DRAWING_LIMIT_RATIO_SUM,CC_CC_MIN_PAYMENT_RATIO_MIN,CC_CC_MIN_PAYMENT_RATIO_MAX,CC_CC_MIN_PAYMENT_RATIO_MEAN,CC_CC_MIN_PAYMENT_RATIO_SUM,CC_RECORD_COUNT,CC_CARD_COUNT,CC_LATE_RATE,CC_SEVERE_DPD_RATE,HAS_CREDIT_CARD_HISTORY
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0.0,NaN,NaN,NaN,0.0,6.0,1.0,0.0,0.0,1
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


In [4]:
# Confirm no duplicate applicants and TARGET is intact
print("Unique applicants:", application_model["SK_ID_CURR"].nunique())
print("Duplicate applicant IDs:", application_model["SK_ID_CURR"].duplicated().sum())
print("TARGET present:", "TARGET" in application_model.columns)
print("Missing TARGET:", application_model["TARGET"].isna().sum())
print("Target distribution:")
display(application_model["TARGET"].value_counts(normalize=True))

Unique applicants: 307511
Duplicate applicant IDs: 0
TARGET present: True
Missing TARGET: 0
Target distribution:


,proportion
TARGET,
0,0.919271
1,0.080729


In [5]:
# Load the locked Model08 benchmark to compare against
model08_benchmark = pd.read_csv(MODEL08_BENCHMARK_PATH)
display(model08_benchmark)

MODEL08_ROC_AUC = float(model08_benchmark.loc[0, "ROC-AUC"])
MODEL08_PR_AUC = float(model08_benchmark.loc[0, "PR-AUC"])

print("Locked Model08 ROC-AUC:", MODEL08_ROC_AUC)
print("Locked Model08 PR-AUC:", MODEL08_PR_AUC)

,Model,ROC-AUC,PR-AUC
0,Model08,0.787,0.2872


Locked Model08 ROC-AUC: 0.787
Locked Model08 PR-AUC: 0.2872


In [6]:
# Split into features and target
X = application_model.drop(columns=["TARGET", "SK_ID_CURR"])
y = application_model["TARGET"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (307511, 400)
y shape: (307511,)


In [7]:
# Same split used throughout the project for clean experiment comparison
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training shape:", X_train.shape)
print("Validation shape:", X_valid.shape)
print("\nTrain target distribution:")
print(y_train.value_counts(normalize=True))
print("\nValidation target distribution:")
print(y_valid.value_counts(normalize=True))

Training shape: (246008, 400)
Validation shape: (61503, 400)

Train target distribution:
TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64

Validation target distribution:
TARGET
0    0.919272
1    0.080728
Name: proportion, dtype: float64


In [8]:
# Separate numeric vs categorical columns
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Numeric features: 384
Categorical features: 16


In [9]:
# Report missingness without automatically removing anything -
# missing historical data can itself be informative
missing_summary = pd.DataFrame({
    "Missing_Count": X_train.isna().sum(),
    "Missing_Percentage": X_train.isna().mean() * 100
})
missing_summary = missing_summary.sort_values("Missing_Percentage", ascending=False)

print("Top 30 features by missingness:")
display(missing_summary.head(30))

Top 30 features by missingness:


,Missing_Count,Missing_Percentage
CC_CC_PAYMENT_RECEIVABLE_RATIO_MIN,198095,80.523804
CC_CC_PAYMENT_RECEIVABLE_RATIO_MEAN,198095,80.523804
CC_CC_PAYMENT_RECEIVABLE_RATIO_MAX,198095,80.523804
CC_CC_MIN_PAYMENT_RATIO_MAX,197212,80.164873
CC_CC_MIN_PAYMENT_RATIO_MEAN,197212,80.164873
CC_CC_MIN_PAYMENT_RATIO_MIN,197212,80.164873
CC_AMT_PAYMENT_CURRENT_MIN,197059,80.102680
CC_AMT_PAYMENT_CURRENT_MAX,197059,80.102680
CC_AMT_PAYMENT_CURRENT_MEAN,197059,80.102680
CC_CNT_DRAWINGS_ATM_CURRENT_MIN,196989,80.074225


In [10]:
# Identify features with only one distinct value
constant_features = []
for col in X_train.columns:
    if X_train[col].nunique(dropna=False) <= 1:
        constant_features.append(col)

print("Constant features:", len(constant_features))
for col in constant_features:
    print("-", col)

Constant features: 0


In [11]:
# Genuine redundancy check, decisions based on train only
duplicate_features = []
cols = X_train.columns.tolist()

for i in range(len(cols)):
    col_1 = cols[i]
    for j in range(i + 1, len(cols)):
        col_2 = cols[j]
        if X_train[col_1].equals(X_train[col_2]):
            duplicate_features.append(col_2)

print("Exact duplicate features:", len(duplicate_features))
for col in duplicate_features:
    print("-", col)

Exact duplicate features: 1
- CC_SK_DPD_DEF_MIN


In [12]:
# Test dropping features that showed weak linear correlation with TARGET in MODEL02 EDA,
# rather than assuming weak correlation means useless
LOW_CORR_CANDIDATES = ["CREDIT_INCOME_RATIO", "ANNUITY_CREDIT_RATIO", "ANNUITY_INCOME_RATIO"]

available_low_corr = [col for col in LOW_CORR_CANDIDATES if col in X_train.columns]

print("Low-correlation candidates available:")
for col in available_low_corr:
    print("-", col)

print("\nTraining-set correlations with TARGET:")
train_numeric_for_corr = X_train[available_low_corr].copy()
train_numeric_for_corr["TARGET"] = y_train.values

display(train_numeric_for_corr.corr()["TARGET"].drop("TARGET").sort_values())

Low-correlation candidates available:
- CREDIT_INCOME_RATIO
- ANNUITY_CREDIT_RATIO
- ANNUITY_INCOME_RATIO

Training-set correlations with TARGET:


,TARGET
CREDIT_INCOME_RATIO,-0.007445
ANNUITY_CREDIT_RATIO,0.013854
ANNUITY_INCOME_RATIO,0.015459


In [13]:
# Rank all numeric features by absolute correlation with TARGET, train-only
numeric_train = X_train.select_dtypes(include=np.number)

target_corr = (
    pd.concat([numeric_train, y_train.rename("TARGET")], axis=1)
    .corr()["TARGET"]
    .drop("TARGET")
    .sort_values(key=np.abs, ascending=False)
)

print("Top 30 numeric features by absolute TARGET correlation:")
display(target_corr.head(30))

Top 30 numeric features by absolute TARGET correlation:


,TARGET
EXT_SOURCE_MEAN,-0.221854
EXT_SOURCE_2_3,-0.198384
EXT_SOURCE_MAX,-0.196980
EXT_SOURCE_1_3,-0.186704
EXT_SOURCE_MIN,-0.185168
EXT_SOURCE_3,-0.178845
EXT_SOURCE_1_2,-0.175034
EXT_SOURCE_2,-0.159593
EXT_SOURCE_1,-0.155362
CC_CC_UTILIZATION_RATIO_MEAN,0.136937


In [14]:
# Identify numeric feature pairs correlated at or above 0.95, without auto-deleting
CORR_THRESHOLD = 0.95

corr_matrix = numeric_train.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

high_corr_pairs = []
for col in upper.columns:
    for row in upper.index:
        value = upper.loc[row, col]
        if pd.notna(value) and value >= CORR_THRESHOLD:
            high_corr_pairs.append((row, col, value))

high_corr_pairs_df = pd.DataFrame(high_corr_pairs, columns=["Feature_1", "Feature_2", "Absolute_Correlation"])
high_corr_pairs_df = high_corr_pairs_df.sort_values("Absolute_Correlation", ascending=False).reset_index(drop=True)

print("Number of highly correlated numeric pairs:", len(high_corr_pairs_df))
display(high_corr_pairs_df.head(50))

Number of highly correlated numeric pairs: 146


,Feature_1,Feature_2,Absolute_Correlation
0,DAYS_EMPLOYED,EMPLOYMENT_YEARS,1.000000
1,POS_POS_REMAINING_INST_RATIO_MAX,POS_POS_INSTALLMENT_PROGRESS_MIN,1.000000
2,DAYS_BIRTH,AGE_YEARS,1.000000
3,POS_NAME_CONTRACT_STATUS_Completed,POS_COMPLETED_RATE,1.000000
4,POS_POS_REMAINING_INST_RATIO_MIN,POS_POS_INSTALLMENT_PROGRESS_MAX,1.000000
5,POS_POS_REMAINING_INST_RATIO_MEAN,POS_POS_INSTALLMENT_PROGRESS_MEAN,1.000000
6,POS_POS_REMAINING_INST_RATIO_VAR,POS_POS_INSTALLMENT_PROGRESS_VAR,1.000000
7,CC_AMT_RECIVABLE_MIN,CC_AMT_TOTAL_RECEIVABLE_MIN,1.000000
8,CC_AMT_RECIVABLE_MEAN,CC_AMT_TOTAL_RECEIVABLE_MEAN,0.999998
9,CC_AMT_RECIVABLE_MAX,CC_AMT_TOTAL_RECEIVABLE_MAX,0.999998


In [15]:
# For each highly correlated pair, keep the feature with stronger target correlation
redundancy_candidates = set()

for _, row in high_corr_pairs_df.iterrows():
    feature_1 = row["Feature_1"]
    feature_2 = row["Feature_2"]

    corr_1 = abs(target_corr.get(feature_1, 0))
    corr_2 = abs(target_corr.get(feature_2, 0))

    if corr_1 >= corr_2:
        redundancy_candidates.add(feature_2)
    else:
        redundancy_candidates.add(feature_1)

print("Potential redundant features:", len(redundancy_candidates))
for col in sorted(redundancy_candidates):
    print("-", col)

Potential redundant features: 97
- AGE_YEARS
- AMT_CREDIT
- APARTMENTS_MEDI
- APARTMENTS_MODE
- BASEMENTAREA_MEDI
- BASEMENTAREA_MODE
- BUREAU_AMT_CREDIT_SUM_DEBT_MAX
- BUREAU_OVERDUE_AMOUNT_MAX
- CC_AMT_INST_MIN_REGULARITY_MAX
- CC_AMT_INST_MIN_REGULARITY_MEAN
- CC_AMT_INST_MIN_REGULARITY_SUM
- CC_AMT_PAYMENT_CURRENT_MAX
- CC_AMT_PAYMENT_CURRENT_MEAN
- CC_AMT_PAYMENT_TOTAL_CURRENT_SUM
- CC_AMT_RECEIVABLE_PRINCIPAL_MAX
- CC_AMT_RECEIVABLE_PRINCIPAL_MEAN
- CC_AMT_RECEIVABLE_PRINCIPAL_MIN
- CC_AMT_RECEIVABLE_PRINCIPAL_SUM
- CC_AMT_RECIVABLE_MAX
- CC_AMT_RECIVABLE_MEAN
- CC_AMT_RECIVABLE_MIN
- CC_AMT_RECIVABLE_SUM
- CC_AMT_TOTAL_RECEIVABLE_MAX
- CC_AMT_TOTAL_RECEIVABLE_MEAN
- CC_AMT_TOTAL_RECEIVABLE_MIN
- CC_AMT_TOTAL_RECEIVABLE_SUM
- CC_CC_MIN_PAYMENT_RATIO_MAX
- CC_CC_PAYMENT_RECEIVABLE_RATIO_SUM
- CC_CC_UTILIZATION_RATIO_SUM
- CC_CNT_DRAWINGS_POS_CURRENT_MAX
- CC_CNT_DRAWINGS_POS_CURRENT_MEAN
- CC_CNT_DRAWINGS_POS_CURRENT_SUM
- CC_CNT_INSTALMENT_MATURE_CUM_MAX
- CC_CNT_INSTALMENT_MATUR

In [16]:
# Check how much overlap exists between the two candidate-removal sets
overlap = set(available_low_corr) & set(redundancy_candidates)

print("Features in BOTH candidate sets:", len(overlap))
for col in sorted(overlap):
    print("-", col)

Features in BOTH candidate sets: 0


In [17]:
# Group features by which table they came from, for interpretability
def feature_group(col):
    if col.startswith("PREV_"):
        return "Previous Application"
    if col.startswith("BUREAU_"):
        return "Bureau"
    if col.startswith("BB_"):
        return "Bureau Balance"
    if col.startswith("POS_"):
        return "POS_CASH"
    if col.startswith("INST_"):
        return "Installments"
    if col.startswith("CC_"):
        return "Credit Card"
    return "Application / Other"

group_summary = pd.Series(X_train.columns.map(feature_group)).value_counts()

print("Feature groups:")
display(group_summary)

Feature groups:


,count
Application / Other,140
Credit Card,100
POS_CASH,53
Previous Application,34
Bureau,31
Installments,26
Bureau Balance,16


In [18]:
# Reusable function to build a fresh preprocessor for any feature subset
def build_preprocessor(X_data):
    numeric_cols = X_data.select_dtypes(include=np.number).columns.tolist()
    categorical_cols = X_data.select_dtypes(include=["object"]).columns.tolist()

    numeric_transformer = Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    return ColumnTransformer(transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ])

In [19]:
# Reusable function to build the same XGBoost config used throughout the project
def build_xgb():
    return XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

In [20]:
# Confirm no infinite values remain anywhere in X_train before training
inf_count = np.isinf(X_train.select_dtypes(include=np.number)).sum().sum()
print("Infinite values in X_train:", inf_count)

Infinite values in X_train: 0


In [21]:
# Reusable function to train and evaluate one feature-set experiment
def run_xgb_experiment(X_train_exp, X_valid_exp, y_train_exp, y_valid_exp, experiment_name):
    print("\n" + "=" * 70)
    print(experiment_name)
    print("=" * 70)
    print("Training features:", X_train_exp.shape[1])

    preprocessor = build_preprocessor(X_train_exp)
    model = build_xgb()

    pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])
    pipeline.fit(X_train_exp, y_train_exp)

    valid_proba = pipeline.predict_proba(X_valid_exp)[:, 1]

    roc = roc_auc_score(y_valid_exp, valid_proba)
    pr = average_precision_score(y_valid_exp, valid_proba)

    print(f"ROC-AUC: {roc:.4f}")
    print(f"PR-AUC:  {pr:.4f}")

    return pipeline, roc, pr

In [22]:
# Critical sanity check: no feature removal, same split, same XGBoost config
baseline_pipeline, baseline_roc_auc, baseline_pr_auc = run_xgb_experiment(
    X_train, X_valid, y_train, y_valid,
    "MODEL09 EXPERIMENT 1 - MODEL08 REPRODUCTION"
)

print("\nLocked Model08 benchmark:")
print(f"ROC-AUC: {MODEL08_ROC_AUC:.4f}")
print(f"PR-AUC:  {MODEL08_PR_AUC:.4f}")

print("\nReproduced result:")
print(f"ROC-AUC: {baseline_roc_auc:.4f}")
print(f"PR-AUC:  {baseline_pr_auc:.4f}")

print("\nDifference:")
print(f"ROC-AUC difference: {baseline_roc_auc - MODEL08_ROC_AUC:+.6f}")
print(f"PR-AUC difference:  {baseline_pr_auc - MODEL08_PR_AUC:+.6f}")


MODEL09 EXPERIMENT 1 - MODEL08 REPRODUCTION
Training features: 400
ROC-AUC: 0.7874
PR-AUC:  0.2863

Locked Model08 benchmark:
ROC-AUC: 0.7870
PR-AUC:  0.2872

Reproduced result:
ROC-AUC: 0.7874
PR-AUC:  0.2863

Difference:
ROC-AUC difference: +0.000437
PR-AUC difference:  -0.000946


In [23]:
# Confirm the parquet round-trip didn't silently change anything
roc_reproduction_ok = abs(baseline_roc_auc - MODEL08_ROC_AUC) < 0.001
pr_reproduction_ok = abs(baseline_pr_auc - MODEL08_PR_AUC) < 0.001

print("ROC-AUC reproduction within tolerance:", roc_reproduction_ok)
print("PR-AUC reproduction within tolerance:", pr_reproduction_ok)

if not roc_reproduction_ok or not pr_reproduction_ok:
    print("\nWARNING: Model08 was not reproduced closely.")
    print("Do NOT interpret the refinement experiments yet.")
else:
    print("\nModel08 parquet round-trip sanity check passed.")

ROC-AUC reproduction within tolerance: True
PR-AUC reproduction within tolerance: True

Model08 parquet round-trip sanity check passed.


In [24]:
# Drop the low-correlation MODEL02 candidates
X_train_lowcorr = X_train.drop(columns=available_low_corr, errors="ignore")
X_valid_lowcorr = X_valid.drop(columns=available_low_corr, errors="ignore")

print("Removed low-correlation features:")
for col in available_low_corr:
    print("-", col)
print("Training feature count:", X_train_lowcorr.shape[1])

Removed low-correlation features:
- CREDIT_INCOME_RATIO
- ANNUITY_CREDIT_RATIO
- ANNUITY_INCOME_RATIO
Training feature count: 397


In [25]:
# Train and evaluate with low-correlation features removed
lowcorr_pipeline, lowcorr_roc_auc, lowcorr_pr_auc = run_xgb_experiment(
    X_train_lowcorr, X_valid_lowcorr, y_train, y_valid,
    "MODEL09 EXPERIMENT 2 - DROP LOW-CORRELATION MODEL02 FEATURES"
)

print("ROC-AUC change vs Model08 baseline:", f"{lowcorr_roc_auc - baseline_roc_auc:+.4f}")
print("PR-AUC change vs Model08 baseline:", f"{lowcorr_pr_auc - baseline_pr_auc:+.4f}")

del lowcorr_pipeline
gc.collect()


MODEL09 EXPERIMENT 2 - DROP LOW-CORRELATION MODEL02 FEATURES
Training features: 397
ROC-AUC: 0.7860
PR-AUC:  0.2867
ROC-AUC change vs Model08 baseline: -0.0014
PR-AUC change vs Model08 baseline: +0.0004


199

In [26]:
# Drop the redundancy candidates identified from high-correlation pairs
X_train_redundancy = X_train.drop(columns=list(redundancy_candidates), errors="ignore")
X_valid_redundancy = X_valid.drop(columns=list(redundancy_candidates), errors="ignore")

print("Redundancy candidates removed:", len(redundancy_candidates))
print("Training feature count:", X_train_redundancy.shape[1])

Redundancy candidates removed: 97
Training feature count: 303


In [27]:
# Train and evaluate with redundant features removed
redundancy_pipeline, redundancy_roc_auc, redundancy_pr_auc = run_xgb_experiment(
    X_train_redundancy, X_valid_redundancy, y_train, y_valid,
    "MODEL09 EXPERIMENT 3 - REDUNDANCY REFINEMENT"
)

print("ROC-AUC change vs Model08 baseline:", f"{redundancy_roc_auc - baseline_roc_auc:+.4f}")
print("PR-AUC change vs Model08 baseline:", f"{redundancy_pr_auc - baseline_pr_auc:+.4f}")
del redundancy_pipeline
gc.collect()


MODEL09 EXPERIMENT 3 - REDUNDANCY REFINEMENT
Training features: 303
ROC-AUC: 0.7872
PR-AUC:  0.2887
ROC-AUC change vs Model08 baseline: -0.0002
PR-AUC change vs Model08 baseline: +0.0025


110

In [28]:
# Remove both low-correlation and redundancy candidates together
combined_remove = set(available_low_corr) | set(redundancy_candidates)

X_train_combined = X_train.drop(columns=list(combined_remove), errors="ignore")
X_valid_combined = X_valid.drop(columns=list(combined_remove), errors="ignore")

print("Total combined removals:", len(combined_remove))
print("Training feature count:", X_train_combined.shape[1])

Total combined removals: 100
Training feature count: 300


In [29]:
# Train and evaluate with both candidate sets removed
combined_pipeline, combined_roc_auc, combined_pr_auc = run_xgb_experiment(
    X_train_combined, X_valid_combined, y_train, y_valid,
    "MODEL09 EXPERIMENT 4 - COMBINED REFINEMENT"
)

print("ROC-AUC change vs Model08 baseline:", f"{combined_roc_auc - baseline_roc_auc:+.4f}")
print("PR-AUC change vs Model08 baseline:", f"{combined_pr_auc - baseline_pr_auc:+.4f}")
del combined_pipeline
gc.collect()


MODEL09 EXPERIMENT 4 - COMBINED REFINEMENT
Training features: 300
ROC-AUC: 0.7854
PR-AUC:  0.2842
ROC-AUC change vs Model08 baseline: -0.0020
PR-AUC change vs Model08 baseline: -0.0021


110

In [30]:
# Build a full comparison table across all four experiments
refinement_results = pd.DataFrame({
    "Experiment": [
        "Model08 Reproduction", "Drop Low-Correlation Features",
        "Redundancy Refinement", "Combined Refinement"
    ],
    "ROC-AUC": [baseline_roc_auc, lowcorr_roc_auc, redundancy_roc_auc, combined_roc_auc],
    "PR-AUC": [baseline_pr_auc, lowcorr_pr_auc, redundancy_pr_auc, combined_pr_auc],
    "ROC-AUC Change vs Model08": [
        baseline_roc_auc - MODEL08_ROC_AUC, lowcorr_roc_auc - MODEL08_ROC_AUC,
        redundancy_roc_auc - MODEL08_ROC_AUC, combined_roc_auc - MODEL08_ROC_AUC
    ],
    "PR-AUC Change vs Model08": [
        baseline_pr_auc - MODEL08_PR_AUC, lowcorr_pr_auc - MODEL08_PR_AUC,
        redundancy_pr_auc - MODEL08_PR_AUC, combined_pr_auc - MODEL08_PR_AUC
    ],
    "Feature Count": [
        X_train.shape[1], X_train_lowcorr.shape[1],
        X_train_redundancy.shape[1], X_train_combined.shape[1]
    ]
})

display(refinement_results.style.format({
    "ROC-AUC": "{:.4f}", "PR-AUC": "{:.4f}",
    "ROC-AUC Change vs Model08": "{:+.4f}", "PR-AUC Change vs Model08": "{:+.4f}"
}))

,Experiment,ROC-AUC,PR-AUC,ROC-AUC Change vs Model08,PR-AUC Change vs Model08,Feature Count
0,Model08 Reproduction,0.7874,0.2863,+0.0004,-0.0009,400
1,Drop Low-Correlation Features,0.7860,0.2867,-0.0010,-0.0005,397
2,Redundancy Refinement,0.7872,0.2887,+0.0002,+0.0015,303
3,Combined Refinement,0.7854,0.2842,-0.0016,-0.0030,300


In [31]:
# Select the best experiment, prioritizing PR-AUC given the class imbalance
candidate_scores = pd.DataFrame({
    "Experiment": [
        "Model08 Reproduction", "Drop Low-Correlation Features",
        "Redundancy Refinement", "Combined Refinement"
    ],
    "ROC-AUC": [baseline_roc_auc, lowcorr_roc_auc, redundancy_roc_auc, combined_roc_auc],
    "PR-AUC": [baseline_pr_auc, lowcorr_pr_auc, redundancy_pr_auc, combined_pr_auc]
})

best_experiment = candidate_scores.sort_values(["PR-AUC", "ROC-AUC"], ascending=False).iloc[0]

print("Best experiment by PR-AUC, then ROC-AUC:")
display(best_experiment.to_frame())

Best experiment by PR-AUC, then ROC-AUC:


,2
Experiment,Redundancy Refinement
ROC-AUC,0.787216
PR-AUC,0.288721


In [32]:
# State the final decision explicitly
print("Model09 recommended experiment:", best_experiment["Experiment"])
print(f"ROC-AUC: {best_experiment['ROC-AUC']:.4f}")
print(f"PR-AUC:  {best_experiment['PR-AUC']:.4f}")
print(f"ROC-AUC vs Model08: {best_experiment['ROC-AUC'] - MODEL08_ROC_AUC:+.4f}")
print(f"PR-AUC vs Model08:  {best_experiment['PR-AUC'] - MODEL08_PR_AUC:+.4f}")

Model09 recommended experiment: Redundancy Refinement
ROC-AUC: 0.7872
PR-AUC:  0.2887
ROC-AUC vs Model08: +0.0002
PR-AUC vs Model08:  +0.0015


In [33]:
# Assign the recommended feature columns; retrain the pipeline since
# earlier experiment pipelines were freed immediately after their metrics were captured
best_name = best_experiment["Experiment"]

if best_name == "Drop Low-Correlation Features":
    recommended_feature_columns = X_train_lowcorr.columns.tolist()
    recommended_pipeline, _, _ = run_xgb_experiment(
        X_train_lowcorr, X_valid_lowcorr, y_train, y_valid, "RETRAIN - RECOMMENDED (Low-Correlation)"
    )
elif best_name == "Redundancy Refinement":
    recommended_feature_columns = X_train_redundancy.columns.tolist()
    recommended_pipeline, _, _ = run_xgb_experiment(
        X_train_redundancy, X_valid_redundancy, y_train, y_valid, "RETRAIN - RECOMMENDED (Redundancy)"
    )
elif best_name == "Combined Refinement":
    recommended_feature_columns = X_train_combined.columns.tolist()
    recommended_pipeline, _, _ = run_xgb_experiment(
        X_train_combined, X_valid_combined, y_train, y_valid, "RETRAIN - RECOMMENDED (Combined)"
    )
else:
    recommended_feature_columns = X_train.columns.tolist()
    recommended_pipeline = baseline_pipeline

print("Recommended feature set:", best_name)
print("Recommended feature count:", len(recommended_feature_columns))


RETRAIN - RECOMMENDED (Redundancy)
Training features: 303
ROC-AUC: 0.7872
PR-AUC:  0.2887
Recommended feature set: Redundancy Refinement
Recommended feature count: 303


In [34]:
# List every feature removed in the recommended set
removed_from_best = sorted(set(X_train.columns) - set(recommended_feature_columns))

print("Total features removed in recommended set:", len(removed_from_best))
for col in removed_from_best:
    print("-", col)

Total features removed in recommended set: 97
- AGE_YEARS
- AMT_CREDIT
- APARTMENTS_MEDI
- APARTMENTS_MODE
- BASEMENTAREA_MEDI
- BASEMENTAREA_MODE
- BUREAU_AMT_CREDIT_SUM_DEBT_MAX
- BUREAU_OVERDUE_AMOUNT_MAX
- CC_AMT_INST_MIN_REGULARITY_MAX
- CC_AMT_INST_MIN_REGULARITY_MEAN
- CC_AMT_INST_MIN_REGULARITY_SUM
- CC_AMT_PAYMENT_CURRENT_MAX
- CC_AMT_PAYMENT_CURRENT_MEAN
- CC_AMT_PAYMENT_TOTAL_CURRENT_SUM
- CC_AMT_RECEIVABLE_PRINCIPAL_MAX
- CC_AMT_RECEIVABLE_PRINCIPAL_MEAN
- CC_AMT_RECEIVABLE_PRINCIPAL_MIN
- CC_AMT_RECEIVABLE_PRINCIPAL_SUM
- CC_AMT_RECIVABLE_MAX
- CC_AMT_RECIVABLE_MEAN
- CC_AMT_RECIVABLE_MIN
- CC_AMT_RECIVABLE_SUM
- CC_AMT_TOTAL_RECEIVABLE_MAX
- CC_AMT_TOTAL_RECEIVABLE_MEAN
- CC_AMT_TOTAL_RECEIVABLE_MIN
- CC_AMT_TOTAL_RECEIVABLE_SUM
- CC_CC_MIN_PAYMENT_RATIO_MAX
- CC_CC_PAYMENT_RECEIVABLE_RATIO_SUM
- CC_CC_UTILIZATION_RATIO_SUM
- CC_CNT_DRAWINGS_POS_CURRENT_MAX
- CC_CNT_DRAWINGS_POS_CURRENT_MEAN
- CC_CNT_DRAWINGS_POS_CURRENT_SUM
- CC_CNT_INSTALMENT_MATURE_CUM_MAX
- CC_CNT_INS

In [35]:
# Save a table documenting which candidates were considered and whether they were kept
candidate_removal_table = pd.DataFrame({
    "Feature": sorted(set(available_low_corr) | set(redundancy_candidates)),
    "Low_Correlation_Candidate": [
        col in available_low_corr for col in sorted(set(available_low_corr) | set(redundancy_candidates))
    ],
    "Redundancy_Candidate": [
        col in redundancy_candidates for col in sorted(set(available_low_corr) | set(redundancy_candidates))
    ],
    "Selected_In_Recommended_Set": [
        col in recommended_feature_columns for col in sorted(set(available_low_corr) | set(redundancy_candidates))
    ]
})

candidate_removal_table.to_csv(RESULTS_PATH + "model09_refinement_candidates.csv", index=False)
print("Refinement candidate table saved.")

Refinement candidate table saved.


In [36]:
# Save the recommended feature list for use in Model10
pd.DataFrame({"Feature": recommended_feature_columns}).to_csv(
    RESULTS_PATH + "model09_recommended_feature_list.csv", index=False
)
print("Recommended feature list saved.")

Recommended feature list saved.


In [37]:
# Save the full experiment comparison table
refinement_results.to_csv(RESULTS_PATH + "model09_refinement_results.csv", index=False)
print("Model09 results saved.")

Model09 results saved.


In [38]:
# Save the high-correlation pair table for reference
high_corr_pairs_df.to_csv(RESULTS_PATH + "model09_high_correlation_pairs.csv", index=False)
print("High-correlation pair table saved.")

High-correlation pair table saved.


In [39]:
# Install MLflow and point at the same tracking database used throughout the project
!pip install mlflow -q
import mlflow

mlflow.set_tracking_uri("sqlite:////content/drive/MyDrive/RupeeRisk/mlflow.db")
mlflow.set_experiment("RupeeRisk")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 94.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.2/216.2 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132

<Experiment: artifact_location='/content/mlruns/1', creation_time=1787393296537, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1787393296537, lifecycle_stage='active', name='RupeeRisk', tags={}, trace_location=None, workspace='default'>

In [40]:
# Record this stage's parameters and metrics
with mlflow.start_run(run_name="XGBoost_Feature_Refinement"):
    mlflow.log_param("stage", "Model09 - Feature Refinement")
    mlflow.log_param("input", "Model08 consolidated parquet")
    mlflow.log_param("model", "XGBoost")
    mlflow.log_param("original_feature_count", int(X_train.shape[1]))
    mlflow.log_param("low_correlation_candidates", ", ".join(available_low_corr))
    mlflow.log_param("n_low_correlation_candidates", len(available_low_corr))
    mlflow.log_param("correlation_threshold", CORR_THRESHOLD)
    mlflow.log_param("n_redundancy_candidates", len(redundancy_candidates))
    mlflow.log_param("recommended_experiment", best_name)
    mlflow.log_param("recommended_feature_count", len(recommended_feature_columns))

    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("subsample", 0.8)
    mlflow.log_param("colsample_bytree", 0.8)

    mlflow.log_metric("model08_baseline_roc_auc", baseline_roc_auc)
    mlflow.log_metric("model08_baseline_pr_auc", baseline_pr_auc)
    mlflow.log_metric("lowcorr_roc_auc", lowcorr_roc_auc)
    mlflow.log_metric("lowcorr_pr_auc", lowcorr_pr_auc)
    mlflow.log_metric("redundancy_roc_auc", redundancy_roc_auc)
    mlflow.log_metric("redundancy_pr_auc", redundancy_pr_auc)
    mlflow.log_metric("combined_roc_auc", combined_roc_auc)
    mlflow.log_metric("combined_pr_auc", combined_pr_auc)

print("Model09 logged to MLflow.")

Model09 logged to MLflow.


In [41]:
# Pull every run logged so far for comparison
runs = mlflow.search_runs(experiment_names=["RupeeRisk"])
display(runs[["tags.mlflow.runName", "metrics.roc_auc", "metrics.pr_auc"]])

,tags.mlflow.runName,metrics.roc_auc,metrics.pr_auc
0,XGBoost_Feature_Refinement,NaN,NaN
1,XGBoost_Credit_Card,0.787019,0.287171
2,XGBoost_Installments,0.785949,0.286362
3,XGBoost_POS_CASH,0.783313,0.278581
4,XGBoost_Bureau,0.777585,0.274161
5,XGBoost_Previous_Application,0.775428,0.265853
6,XGBoost_Application_Features,0.769403,0.262725
7,XGBoost_scale_pos_weight,0.760000,0.249300
8,XGBoost_Baseline,0.761200,0.251600
9,Logistic_Regression_Baseline,0.750100,0.232600


In [42]:
# Print the overall summary of Model09's refinement decision
print("""
MODEL09 FEATURE REFINEMENT COMPLETE

Locked Model08 benchmark: ROC-AUC = {:.4f}, PR-AUC = {:.4f}
Model09 reproduction:     ROC-AUC = {:.4f}, PR-AUC = {:.4f}

Best refinement: {}
Best refined ROC-AUC: {:.4f}
Best refined PR-AUC:  {:.4f}

Features before refinement: {}
Recommended features: {}
Features removed: {}

Next: Polynomial/domain feature experiment, Optuna XGBoost tuning,
final model selection, SHAP explainability, final evaluation.
""".format(
    MODEL08_ROC_AUC, MODEL08_PR_AUC,
    baseline_roc_auc, baseline_pr_auc,
    best_name,
    float(best_experiment["ROC-AUC"]), float(best_experiment["PR-AUC"]),
    X_train.shape[1], len(recommended_feature_columns), len(removed_from_best)
))


MODEL09 FEATURE REFINEMENT COMPLETE

Locked Model08 benchmark: ROC-AUC = 0.7870, PR-AUC = 0.2872
Model09 reproduction:     ROC-AUC = 0.7874, PR-AUC = 0.2863

Best refinement: Redundancy Refinement
Best refined ROC-AUC: 0.7872
Best refined PR-AUC:  0.2887

Features before refinement: 400
Recommended features: 303
Features removed: 97

Next: Polynomial/domain feature experiment, Optuna XGBoost tuning,
final model selection, SHAP explainability, final evaluation.



In [43]:
# Remove intermediate experiment matrices no longer needed
del X_train_lowcorr, X_valid_lowcorr
del X_train_redundancy, X_valid_redundancy
del X_train_combined, X_valid_combined

gc.collect()
print("Model09 intermediate matrices cleaned from memory.")
!free -h

Model09 intermediate matrices cleaned from memory.
               total        used        free      shared  buff/cache   available
Mem:            12Gi       5.1Gi       4.5Gi       3.0Mi       3.0Gi       7.2Gi
Swap:             0B          0B          0B
